# Stage 2 - Pose Extraction

Extracts a 97-frame pose sequence around every one of the 1,457 strokes, for **both** players, and derives handedness from the result.

Everything downstream is a transform of this cache. **Extract once, never repeat** - pose quality is the substrate, and the model cannot recover information the extractor lost.

### Pipeline per stroke

```
seek to win_start  ->  read 97 frames sequentially
                   ->  detector on every 8th frame  ->  interpolate boxes
                   ->  RTMPose-l on each frame (2 crops)
                   ->  float16 npz keyed by stroke_id
```

### Outputs

| file | contents |
|---|---|
| `derived/pose/{video_id}.npz` | keypoints, scores, boxes, table box - one per video |
| `derived/meta/players.json` | handedness derived from wrist-velocity asymmetry |
| `derived/meta/pose_quality.csv` | per-video confidence report |

### Requirements

- **T4 GPU** runtime
- ~2.5 h - checkpointed per video, so a disconnect costs at most one video

> **Resume:** re-running skips any video whose `.npz` already exists. Safe to run repeatedly.


## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install

`rtmlib` declares CPU `onnxruntime` as a hard dependency, which silently shadows `onnxruntime-gpu` and drops you to CPU inference. Installing it `--no-deps` and putting the GPU wheel last is what avoids that.

**Restart the session after this cell**, then continue from cell 3.

In [2]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4."
print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip uninstall -y -q onnxruntime onnxruntime-gpu 2>&1 | tail -1
!pip install -q --no-deps rtmlib 2>&1 | tail -1
!pip install -q opencv-python numpy tqdm ultralytics pyarrow 2>&1 | tail -1
!pip install -q "onnxruntime-gpu==1.22.0" 2>&1 | tail -1

print("\ninstalled:")
!pip list 2>/dev/null | grep -iE "onnxruntime|rtmlib"

import os, glob, site
libs = []
for sp in site.getsitepackages():
    libs += glob.glob(os.path.join(sp, "nvidia", "*", "lib"))
if libs:
    open("/content/_ort_libpath.txt", "w").write(":".join(libs))
    print(f"\nsaved {len(libs)} nvidia lib paths")

print("\n" + "=" * 60)
print("  NOW: Runtime > Restart session, then run from cell 3.")
print("=" * 60)

GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.3 MB/s eta 0:00:00

installed:
onnxruntime-gpu                       1.22.0
rtmlib                                0.0.16

saved 17 nvidia lib paths

  NOW: Runtime > Restart session, then run from cell 3.


## 3. Config

In [3]:
BASE = "/content/drive/MyDrive/tt_coach"

# --- extraction knobs --------------------------------------------------------
DET_STRIDE   = 8       # run the detector every N frames, interpolate between
DET_CONF     = 0.35
BOX_PAD      = 0.18    # expand player box (arm + bat reach beyond the body)
COPY_LOCAL   = True    # copy each video to local disk before reading
                       # random access over the Drive FUSE mount is very slow
SKIP_EXISTING = True   # resume-safe

import os, json, shutil, time
from pathlib import Path

# re-apply CUDA lib paths saved before the restart
_lp = Path("/content/_ort_libpath.txt")
if _lp.exists():
    os.environ["LD_LIBRARY_PATH"] = _lp.read_text() + ":" + os.environ.get("LD_LIBRARY_PATH", "")

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

BASE     = Path(BASE)
META     = BASE / "derived/meta"
VIDEODIR = BASE / "raw/videos"
POSEDIR  = BASE / "derived/pose"
POSEDIR.mkdir(parents=True, exist_ok=True)
LOCAL    = Path("/content/_work"); LOCAL.mkdir(exist_ok=True)

def load(stem):
    p = META / f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META / f"{stem}.csv")

strokes = load("strokes")
folds   = json.loads((META / "folds.json").read_text())
W       = folds["window"]
NF      = W["n_frames"]          # 97
PRE     = W["pre"]               # 60

VIDEO_IDS = sorted(strokes.video_id.unique(),
                   key=lambda v: (v.split("_")[0], int(v.split("_")[1])))

# COCO-17 indices
NOSE = 0
L_SHO, R_SHO = 5, 6
L_ELB, R_ELB = 7, 8
L_WRI, R_WRI = 9, 10
L_HIP, R_HIP = 11, 12

print(f"{len(strokes)} strokes x {NF} frames x 2 players "
      f"= {len(strokes)*NF*2:,} pose inferences")
print(f"window: -{PRE} / +{W['post']} @ {W['native_fps']}fps")

1457 strokes x 97 frames x 2 players = 282,658 pose inferences
window: -60 / +36 @ 120fps


## 4. Load models

In [4]:
import torch
assert torch.cuda.is_available(), "No GPU."

import onnxruntime as ort
provs = ort.get_available_providers()
print(f"onnxruntime {ort.__version__}: {provs}")
assert "CUDAExecutionProvider" in provs, (
    "CUDAExecutionProvider missing - re-run cell 2 and RESTART the session.")

from ultralytics import YOLO
from rtmlib import RTMPose

det = YOLO(str(BASE / "models/detector/best.pt")); det.to("cuda")
PLAYER_CLS = [k for k, v in det.names.items() if v.lower() == "player"][0]
TABLE_CLS  = [k for k, v in det.names.items() if v.lower() == "table"][0]
print(f"detector: {det.names}")

RTM_URL = ("https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/"
           "onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-"
           "3f5a1437_20230504.zip")
pose = RTMPose(onnx_model=RTM_URL, model_input_size=(288, 384),
               backend="onnxruntime", device="cuda")
print("RTMPose-l ready")

onnxruntime 1.22.0: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip


detector: {0: 'player', 1: 'table'}


100%|██████████| 98.9M/98.9M [00:03<00:00, 29.2MB/s]


load /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.onnx with onnxruntime backend
RTMPose-l ready


## 5. Helpers

**Why the detector runs on a stride.** Pose needs a player box for every one of the 141,329 frames, but running detection on all of them would roughly double the extraction cost. Players move smoothly over an 0.8 s window, so detecting every 8th frame and linearly interpolating the box corners gives a tight tracking box for a fraction of the cost.

**Which box is which player.** The detector finds people but not identities, and the umpire is usually in frame too. The table's horizontal centre splits the image; the player box furthest left of it and the one furthest right are the two athletes - the umpire sits near the centre.

In [5]:
def detect_players(frames, mid_x):
    """Batched detection -> list of {'left': box|None, 'right': box|None}."""
    out = []
    res = det.predict(frames, verbose=False, conf=DET_CONF)
    for r in res:
        d = {"left": None, "right": None}
        if r.boxes is not None and len(r.boxes):
            xyxy = r.boxes.xyxy.cpu().numpy()
            cls  = r.boxes.cls.cpu().numpy().astype(int)
            pl   = xyxy[cls == PLAYER_CLS]
            if len(pl):
                cx = (pl[:, 0] + pl[:, 2]) / 2
                ls, rs = pl[cx < mid_x], pl[cx >= mid_x]
                if len(ls):
                    d["left"] = ls[np.argmin((ls[:, 0] + ls[:, 2]) / 2)]
                if len(rs):
                    d["right"] = rs[np.argmax((rs[:, 0] + rs[:, 2]) / 2)]
        out.append(d)
    return out


def interp_boxes(dets, idxs, n):
    """Detections at `idxs` -> a box for every frame 0..n-1.
    Linear interpolation between detections, carry-forward at the ends."""
    known = [(i, b) for i, b in zip(idxs, dets) if b is not None]
    if not known:
        return np.zeros((n, 4), np.float32), np.zeros(n, bool)

    ki = np.array([k[0] for k in known], float)
    kb = np.stack([k[1] for k in known]).astype(float)
    out = np.stack([np.interp(np.arange(n), ki, kb[:, c]) for c in range(4)], 1)
    return out.astype(np.float32), np.ones(n, bool)


def pad_box(b, W_, H_, pad=BOX_PAD):
    x1, y1, x2, y2 = b
    w, h = x2 - x1, y2 - y1
    return np.array([max(0, x1 - w * pad), max(0, y1 - h * pad * 0.6),
                     min(W_, x2 + w * pad), min(H_, y2 + h * pad * 0.25)],
                    np.float32)


def find_table(cap, n_frames, n=9):
    """Table is static - take the median box over sampled frames."""
    boxes = []
    for f in np.linspace(n_frames * 0.1, n_frames * 0.9, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f))
        ok, fr = cap.read()
        if not ok:
            continue
        r = det.predict(fr, verbose=False, conf=DET_CONF)[0]
        if r.boxes is None or not len(r.boxes):
            continue
        xyxy = r.boxes.xyxy.cpu().numpy()
        cls  = r.boxes.cls.cpu().numpy().astype(int)
        tb = xyxy[cls == TABLE_CLS]
        if len(tb):
            boxes.append(tb[np.argmax((tb[:, 2] - tb[:, 0]) * (tb[:, 3] - tb[:, 1]))])
    if not boxes:
        return None
    return np.median(np.stack(boxes), 0).astype(np.float32)

print("helpers ready")

helpers ready


## 6. Extraction

Frames are read **sequentially** after a single seek per stroke - seeking a 120 fps h264 file is expensive, sequential reads are not. Each video is copied to local disk first, because random access over the Drive FUSE mount is dramatically slower than local.

Checkpointed per video. Interrupt and re-run freely.

In [6]:
def extract_video(vid):
    src = VIDEODIR / f"{vid}.mp4"
    out_path = POSEDIR / f"{vid}.npz"
    if SKIP_EXISTING and out_path.exists():
        print(f"  {vid}: exists, skipped"); return

    sv = strokes[strokes.video_id == vid].sort_values("frame_120").reset_index(drop=True)
    if not len(sv):
        return

    path = src
    if COPY_LOCAL:
        path = LOCAL / f"{vid}.mp4"
        if not path.exists():
            t0 = time.time()
            shutil.copy(src, path)
            print(f"  {vid}: copied {src.stat().st_size/1e9:.1f} GB "
                  f"in {time.time()-t0:.0f}s")

    cap = cv2.VideoCapture(str(path))
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    H_, W_ = (int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
              int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))

    table = find_table(cap, n_frames)
    mid_x = (table[0] + table[2]) / 2 if table is not None else W_ / 2

    n = len(sv)
    KP  = np.zeros((n, 2, NF, 17, 2), np.float16)
    SC  = np.zeros((n, 2, NF, 17),    np.float16)
    BX  = np.zeros((n, 2, NF, 4),     np.float16)
    OK  = np.zeros((n, 2), bool)

    det_idx = list(range(0, NF, DET_STRIDE))
    if det_idx[-1] != NF - 1:
        det_idx.append(NF - 1)

    for si, s in tqdm(sv.iterrows(), total=n, desc=f"  {vid}", leave=False):
        start = int(s.win_start)
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, start))

        frames = []
        for k in range(NF):
            idx = start + k
            if idx < 0 or idx >= n_frames:
                frames.append(frames[-1] if frames else
                              np.zeros((H_, W_, 3), np.uint8))   # edge replicate
                continue
            ok, fr = cap.read()
            frames.append(fr if ok else
                          (frames[-1] if frames else np.zeros((H_, W_, 3), np.uint8)))

        # --- boxes: detect on a stride, interpolate the rest ---
        dets = detect_players([frames[i] for i in det_idx], mid_x)
        for pi, side in enumerate(["left", "right"]):
            boxes, valid = interp_boxes([d[side] for d in dets], det_idx, NF)
            if not valid.any():
                continue
            OK[si, pi] = True
            boxes = np.stack([pad_box(b, W_, H_) for b in boxes])
            BX[si, pi] = boxes

        # --- pose, one call per frame with both boxes ---
        for k, fr in enumerate(frames):
            bb, who = [], []
            for pi in range(2):
                if OK[si, pi]:
                    bb.append(BX[si, pi, k].astype(np.float32)); who.append(pi)
            if not bb:
                continue
            kp, sc = pose(fr, bboxes=np.stack(bb))
            for j, pi in enumerate(who):
                KP[si, pi, k] = kp[j]
                SC[si, pi, k] = sc[j]

        del frames

    cap.release()
    if COPY_LOCAL and path != src:
        path.unlink(missing_ok=True)

    np.savez_compressed(
        out_path,
        stroke_id=sv.stroke_id.values.astype(str),
        frame_120=sv.frame_120.values,
        side=sv.side.values.astype(str),
        shot_class=sv.shot_class.values.astype(str),
        keypoints=KP, scores=SC, boxes=BX, detected=OK,
        table_box=table if table is not None else np.zeros(4, np.float32),
        players=np.array(["left", "right"]),
    )
    wrists = SC[..., [L_WRI, R_WRI]]
    print(f"  {vid}: {n} strokes  det={OK.mean():.1%}  "
          f"wrist_conf={wrists[wrists > 0].mean():.3f}  -> {out_path.name}")


t_start = time.time()
for vid in VIDEO_IDS:
    extract_video(vid)
print(f"\ntotal {(time.time()-t_start)/60:.1f} min")

  game_1: copied 5.6 GB in 146s


  game_1:   0%|          | 0/161 [00:00<?, ?it/s]

  game_1: 161 strokes  det=98.8%  wrist_conf=0.803  -> game_1.npz
  game_2: copied 10.8 GB in 317s


  game_2:   0%|          | 0/399 [00:00<?, ?it/s]

  game_2: 399 strokes  det=94.9%  wrist_conf=0.783  -> game_2.npz
  game_3: copied 4.6 GB in 154s


...

  test_5: 27 strokes  det=94.4%  wrist_conf=0.780  -> test_5.npz
  test_6: copied 0.7 GB in 20s


  test_6:   0%|          | 0/39 [00:00<?, ?it/s]

  test_6: 39 strokes  det=94.9%  wrist_conf=0.780  -> test_6.npz
  test_7: copied 0.5 GB in 17s


  test_7:   0%|          | 0/49 [00:00<?, ?it/s]

  test_7: 49 strokes  det=98.0%  wrist_conf=0.800  -> test_7.npz

total 162.0 min


## 7. Quality gate

Wrists are the highest-value and lowest-confidence joint in this whole project - attack-vs-control lives in peak wrist velocity. Measuring wrist confidence explicitly is the check that matters.

**Gate: mean wrist confidence > 0.75 during strokes.**

In [7]:
rows = []
for vid in VIDEO_IDS:
    p = POSEDIR / f"{vid}.npz"
    if not p.exists():
        continue
    d = np.load(p, allow_pickle=True)
    sc, ok, sides = d["scores"], d["detected"], d["side"]

    # striker only - the player who actually played the stroke
    st = np.where(sides == "left", 0, 1)
    ix = np.arange(len(st))
    s_sc, s_ok = sc[ix, st], ok[ix, st]

    wr = s_sc[..., [L_WRI, R_WRI]]
    rows.append({
        "video_id": vid, "n_strokes": len(st),
        "striker_detected": float(s_ok.mean()),
        "wrist_conf": float(wr[wr > 0].mean()) if (wr > 0).any() else 0.0,
        "elbow_conf": float(s_sc[..., [L_ELB, R_ELB]].mean()),
        "shoulder_conf": float(s_sc[..., [L_SHO, R_SHO]].mean()),
        "hip_conf": float(s_sc[..., [L_HIP, R_HIP]].mean()),
        "all_conf": float(s_sc.mean()),
    })

q = pd.DataFrame(rows)
q.to_csv(META / "pose_quality.csv", index=False)
print(q.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\n" + "=" * 62)
mw = q.wrist_conf.mean()
print(f"  mean wrist confidence : {mw:.3f}   {'PASS' if mw > 0.75 else 'BELOW GATE'}")
print(f"  striker detected      : {q.striker_detected.mean():.1%}")
weak = q[q.wrist_conf < 0.70]
if len(weak):
    print(f"\n  weak videos: {list(weak.video_id)}")
    print("  -> inspect crops before trusting these folds")
print("=" * 62)

video_id  n_strokes  striker_detected  wrist_conf  elbow_conf  shoulder_conf  hip_conf  all_conf
  game_1        161             1.000       0.807       0.817          0.787     0.724     0.822
  game_2        399             0.970       0.789       0.774          0.761     0.684     0.780
  game_3        153             0.993       0.833       0.822          0.783     0.714     0.819
  game_4        173             0.983       0.822       0.815          0.793     0.720     0.817
  game_5        248             1.000       0.834       0.848          0.811     0.758     0.855
  test_1         84             1.000       0.829       0.813          0.787     0.723     0.820
  test_2         29             1.000       0.810       0.819          0.797     0.723     0.835
  test_3         24             1.000       0.787       0.780          0.751     0.658     0.781
  test_4         71             1.000       0.787       0.812          0.792     0.713     0.811
  test_5         27           

## 8. Derive handedness

`side` says which *end of the table* a player stands at - independent of which hand holds the bat. A left-hander at the right end produces a mirror-image skeleton to a right-hander at the same end, so Stage 3 has to mirror them.

**The signal.** Across a whole match, the playing arm moves far more and far faster than the free arm. Comparing peak wrist speed (torso-normalised, so it's scale-invariant) between the two wrists gives a large, unambiguous margin - averaged over hundreds of strokes rather than three cropped frames.

Only confident keypoints count, so occlusion doesn't skew it.

In [13]:
def wrist_stats(kp, sc, conf=0.5):
    """-> (left_score, right_score) using torso-normalised peak wrist speed."""
    torso = np.linalg.norm(
        (kp[:, L_SHO] + kp[:, R_SHO]) / 2 - (kp[:, L_HIP] + kp[:, R_HIP]) / 2,
        axis=-1)
    torso = np.median(torso[torso > 1]) if (torso > 1).any() else 1.0

    out = []
    for w in (L_WRI, R_WRI):
        xy, s = kp[:, w].astype(np.float32), sc[:, w]
        good = s > conf
        if good.sum() < 10:
            out.append(0.0); continue
        v = np.linalg.norm(np.diff(xy, axis=0), axis=-1)
        vg = v[good[:-1] & good[1:]]
        out.append(float(np.percentile(vg, 95) / torso) if len(vg) else 0.0)
    return out


players, summary = {}, []
for vid in VIDEO_IDS:
    p = POSEDIR / f"{vid}.npz"
    if not p.exists():
        continue
    d = np.load(p, allow_pickle=True)
    KP, SC, OKD, sides = d["keypoints"], d["scores"], d["detected"], d["side"]

    for pi, side in enumerate(["left", "right"]):
        rows_ = np.where((sides == side) & OKD[:, pi])[0]
        L, R = [], []
        for i in rows_:
            l, r = wrist_stats(KP[i, pi].astype(np.float32), SC[i, pi].astype(np.float32))
            if l > 0 and r > 0:
                L.append(l); R.append(r)

        if len(L) < 3:
            hand, margin = "R", 0.0
        else:
            mL, mR = float(np.mean(L)), float(np.mean(R))
            hand = "L" if mL > mR else "R"
            margin = abs(mL - mR) / max(mL, mR)

        sv = strokes[(strokes.video_id == vid) & (strokes.side == side)]
        key = f"{vid}__{side}"
        players[key] = {
            "video_id": vid, "side": side,
            "handedness": hand,
            "margin": round(margin, 3),
            "n_samples": len(L),
            "fold": folds["video2fold"][vid],
            "n_strokes": int(len(sv)),
            "class_counts": sv.shot_class.value_counts().to_dict(),
            "method": "wrist_velocity_p95" if len(L) >= 3 else "default_no_data",
            "note": "",
        }
        summary.append({"player": key, "fold": folds["video2fold"][vid],
                        "hand": hand, "margin": round(margin, 3),
                        "n": len(L), "strokes": len(sv)})

sm = pd.DataFrame(summary)
print(sm.to_string(index=False))

n_left = (sm.hand == "L").sum()
low = sm[sm.margin < 0.10]
print("\n" + "=" * 62)
print(f"  left-handed: {n_left} / {len(sm)}")
print(f"  (source paper reports 3, all in test videos)")
if len(low):
    print(f"\n  LOW MARGIN (<0.10) - verify these by eye:")
    print(low.to_string(index=False))
print("=" * 62)

       player fold hand  margin   n  strokes
 game_1__left    A    R   0.355  75       75
game_1__right    A    R   0.530  86       86
 game_2__left    B    R   0.452 199      206
game_2__right    B    R   0.453 188      193
 game_3__left    C    R   0.507  81       81
game_3__right    C    R   0.400  71       72
 game_4__left    D    R   0.300  80       82
game_4__right    D    R   0.415  89       91
 game_5__left    E    R   0.199 124      124
game_5__right    E    R   0.411 124      124
 test_1__left    F    R   0.312  44       44
test_1__right    F    R   0.430  40       40
 test_2__left    G    R   0.330  10       10
test_2__right    G    R   0.681  19       19
 test_3__left    G    R   0.501  14       14
test_3__right    G    L   0.443  10       10
 test_4__left    F    R   0.322  33       33
test_4__right    F    R   0.390  38       38
 test_5__left    G    L   0.364  11       11
test_5__right    G    L   0.079  13       16
 test_6__left    G    R   0.396  21       21
test_6__ri

### Review & write

Check the table above. Expect ~3 left-handers, all in `test_*`, each with a healthy margin.

Any player with **margin < 0.10** is a genuine coin-flip - override it in `OVERRIDE` below after looking at the video. Everything else can stand.

In [17]:
OVERRIDE = {}

for k, v in OVERRIDE.items():
    assert k in players, f"unknown player key: {k}"
    players[k]["handedness"] = v.upper()
    players[k]["method"] = "manual_override"

(META / "players.json").write_text(json.dumps(players, indent=2))

lefties = {k: v for k, v in players.items() if v["handedness"] == "L"}
print(f"{len(players)} players, {len(lefties)} left-handed\n")
for k, v in lefties.items():
    print(f"  {k:<20} fold {v['fold']}  margin {v['margin']:.3f}  "
          f"{v['n_strokes']} strokes  [{v['method']}]")

tr = [k for k in lefties if k.startswith("game")]
if tr:
    print(f"\n  note: {len(tr)} left-hander(s) in TRAINING videos: {tr}")
    print("  the paper reports all left-handers in test videos - worth a look")

print(f"\n-> {META / 'players.json'}")

24 players, 12 left-handed

  game_3__right        fold C  margin 0.400  72 strokes  [manual_override]
  game_4__left         fold D  margin 0.300  82 strokes  [manual_override]
  game_4__right        fold D  margin 0.415  91 strokes  [manual_override]
  game_5__left         fold E  margin 0.199  124 strokes  [manual_override]
  test_1__right        fold F  margin 0.430  40 strokes  [manual_override]
  test_3__left         fold G  margin 0.501  14 strokes  [manual_override]
  test_3__right        fold G  margin 0.443  10 strokes  [wrist_velocity_p95]
  test_4__left         fold F  margin 0.322  33 strokes  [manual_override]
  test_5__left         fold G  margin 0.364  11 strokes  [wrist_velocity_p95]
  test_5__right        fold G  margin 0.079  16 strokes  [wrist_velocity_p95]
  test_6__left         fold G  margin 0.396  21 strokes  [manual_override]
  test_7__left         fold G  margin 0.489  25 strokes  [manual_override]

  note: 4 left-hander(s) in TRAINING videos: ['game_3__right'

## 9. Visual verification

Ten random strokes with the skeleton drawn on. This is the cheapest possible check that keypoints land on the right body parts and that the striker was correctly identified.

In [10]:
SKEL = [(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),
        (11,13),(13,15),(12,14),(14,16),(0,5),(0,6)]
try:
    from google.colab.patches import cv2_imshow
except ImportError:
    cv2_imshow = None

sample = strokes.sample(min(10, len(strokes)), random_state=7)
for vid, grp in sample.groupby("video_id"):
    p = POSEDIR / f"{vid}.npz"
    src = VIDEODIR / f"{vid}.mp4"
    if not p.exists() or not src.exists():
        continue
    d = np.load(p, allow_pickle=True)
    ids = list(d["stroke_id"])
    cap = cv2.VideoCapture(str(src))

    for _, s in grp.iterrows():
        if s.stroke_id not in ids:
            continue
        i  = ids.index(s.stroke_id)
        pi = 0 if s.side == "left" else 1
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(s.frame_120))
        ok, fr = cap.read()
        if not ok:
            continue

        kp = d["keypoints"][i, pi, PRE].astype(float)
        sc = d["scores"][i, pi, PRE].astype(float)
        bx = d["boxes"][i, pi, PRE].astype(int)
        cv2.rectangle(fr, (bx[0], bx[1]), (bx[2], bx[3]), (0, 200, 255), 2)
        for a, b in SKEL:
            if sc[a] > .3 and sc[b] > .3:
                cv2.line(fr, tuple(kp[a].astype(int)), tuple(kp[b].astype(int)),
                         (0, 255, 0), 2)
        for j in range(17):
            if sc[j] > .3:
                c = (0, 0, 255) if j in (L_WRI, R_WRI) else (255, 200, 0)
                cv2.circle(fr, tuple(kp[j].astype(int)), 5, c, -1)

        pad = 190
        x1, y1 = max(0, bx[0]-pad), max(0, bx[1]-pad)
        x2, y2 = min(fr.shape[1], bx[2]+pad), min(fr.shape[0], bx[3]+pad)
        crop_ = fr[y1:y2, x1:x2]
        txt = f"{s.stroke_id}  {s.side} {s.technique} -> {s.shot_class.upper()}"
        cv2.putText(crop_, txt, (8, 26), cv2.FONT_HERSHEY_SIMPLEX, .6, (0,0,0), 4)
        cv2.putText(crop_, txt, (8, 26), cv2.FONT_HERSHEY_SIMPLEX, .6, (0,255,255), 1)

        h = 420
        crop_ = cv2.resize(crop_, (int(h * crop_.shape[1] / crop_.shape[0]), h))
        if cv2_imshow:
            cv2_imshow(crop_)
        else:
            from IPython.display import Image, display
            _, b_ = cv2.imencode(".jpg", crop_); display(Image(data=b_.tobytes()))
    cap.release()

print("red = wrists (the joint that matters most)")

Output hidden; open in https://colab.research.google.com to view.

---
## Done

| artifact | used by |
|---|---|
| `derived/pose/*.npz` | Stage 3 - canonicalisation |
| `derived/meta/players.json` | Stage 3 - handedness mirroring |
| `derived/meta/pose_quality.csv` | quality reference |

**Before moving on, confirm:**
1. Mean wrist confidence **> 0.75**
2. Striker detected on **> 98%** of strokes
3. Roughly **3 left-handers**, all in `test_*`, with healthy margins
4. Skeletons in cell 9 land on the right body parts

Next: **`03_canonicalize.ipynb`** - hip-centring, torso scaling, side and handedness mirroring.


In [11]:
# =============================================================================
# CELL 10 - HANDEDNESS VERIFICATION
#
# Shows each candidate at the frame of PEAK WRIST SPEED - the most extended,
# most dynamic moment of the swing, where the bat arm is unmistakable.
#
#     LEFT arm  drawn in MAGENTA   (shoulder -> elbow -> wrist)
#     RIGHT arm drawn in CYAN
#
# The bat arm is the one extended away from the body and reaching furthest.
# Whichever colour that is, is the player's handedness.
#
# Self-contained - safe to run after a restart.
# =============================================================================

BASE = "/content/drive/MyDrive/tt_coach"

# Players to check: the 4 left-handed calls + 2 known right-handers as controls.
# If the controls don't read clearly as R, the visualisation is the problem,
# not the derivation.
VERIFY = [
    "test_3__right",   # L, margin 0.443
    "test_5__left",    # L, margin 0.364   (worst-quality video)
    "test_5__right",   # L, margin 0.079   <-- the suspect
    "test_7__right",   # L, margin 0.249
    "game_1__right",   # R, margin 0.530   control
    "test_1__right",   # R, margin 0.430   control (the chopper)
]
N_SHOW = 3   # strokes per player

import json
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

BASE     = Path(BASE)
META     = BASE / "derived/meta"
POSEDIR  = BASE / "derived/pose"
VIDEODIR = BASE / "raw/videos"

L_SHO, R_SHO = 5, 6
L_ELB, R_ELB = 7, 8
L_WRI, R_WRI = 9, 10
L_HIP, R_HIP = 11, 12

MAGENTA, CYAN = (255, 0, 255), (255, 255, 0)

players = json.loads((META / "players.json").read_text())
folds   = json.loads((META / "folds.json").read_text())
PRE     = folds["window"]["pre"]

try:
    from google.colab.patches import cv2_imshow
except ImportError:
    cv2_imshow = None


def wrist_speed(kp, sc, conf=0.5):
    """Per-frame speed of each wrist, torso-normalised. NaN where unreliable."""
    torso = np.linalg.norm(
        (kp[:, L_SHO] + kp[:, R_SHO]) / 2 - (kp[:, L_HIP] + kp[:, R_HIP]) / 2, axis=-1)
    torso = np.median(torso[torso > 1]) if (torso > 1).any() else 1.0
    out = {}
    for name, w in (("L", L_WRI), ("R", R_WRI)):
        xy = kp[:, w].astype(np.float32)
        v = np.r_[0, np.linalg.norm(np.diff(xy, axis=0), axis=-1)] / torso
        v[sc[:, w] < conf] = np.nan
        out[name] = v
    return out


def draw(frame, kp, sc, thr=0.3):
    """Skeleton with the two arms colour-coded."""
    for a, b in [(5, 6), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15),
                 (12, 14), (14, 16)]:
        if sc[a] > thr and sc[b] > thr:
            cv2.line(frame, tuple(kp[a].astype(int)), tuple(kp[b].astype(int)),
                     (110, 110, 110), 2)
    for (s, e, w), col, tag in ((( L_SHO, L_ELB, L_WRI), MAGENTA, "L"),
                                (( R_SHO, R_ELB, R_WRI), CYAN,    "R")):
        for a, b in ((s, e), (e, w)):
            if sc[a] > thr and sc[b] > thr:
                cv2.line(frame, tuple(kp[a].astype(int)),
                         tuple(kp[b].astype(int)), col, 4)
        if sc[w] > thr:
            p = tuple(kp[w].astype(int))
            cv2.circle(frame, p, 11, col, -1)
            cv2.circle(frame, p, 11, (0, 0, 0), 2)
            cv2.putText(frame, tag, (p[0] - 6, p[1] + 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)
    return frame


def banner(w, text, h=34, colour=(0, 255, 255), scale=0.62):
    b = np.full((h, w, 3), 22, np.uint8)
    cv2.putText(b, text, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 4)
    cv2.putText(b, text, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, scale, colour, 1)
    return b


evidence = []

for key in VERIFY:
    if key not in players:
        print(f"  {key}: not in players.json, skipped"); continue
    info = players[key]
    vid, side = info["video_id"], info["side"]
    pi = 0 if side == "left" else 1

    npz = POSEDIR / f"{vid}.npz"
    src = VIDEODIR / f"{vid}.mp4"
    if not npz.exists() or not src.exists():
        print(f"  {key}: missing files, skipped"); continue

    d   = np.load(npz, allow_pickle=True)
    ids = list(d["stroke_id"])
    rows = np.where((d["side"] == side) & d["detected"][:, pi])[0]

    # prefer attacking strokes: biggest swing, clearest bat arm
    cls = d["shot_class"]
    pref = [i for i in rows if cls[i] == "attack"] or list(rows)
    pick = pref[:N_SHOW * 3][::max(1, len(pref) // max(N_SHOW, 1))][:N_SHOW]

    cap = cv2.VideoCapture(str(src))
    tiles, per_stroke = [], []

    for i in pick:
        kp = d["keypoints"][i, pi].astype(np.float32)
        sc = d["scores"][i, pi].astype(np.float32)
        bx = d["boxes"][i, pi].astype(np.float32)

        sp = wrist_speed(kp, sc)
        pL = np.nanpercentile(sp["L"], 95) if np.isfinite(sp["L"]).any() else np.nan
        pR = np.nanpercentile(sp["R"], 95) if np.isfinite(sp["R"]).any() else np.nan
        per_stroke.append((ids[i], pL, pR))

        # frame where the faster wrist peaks - most extended moment
        fast = sp["L"] if (np.nan_to_num(pL) >= np.nan_to_num(pR)) else sp["R"]
        k = int(np.nanargmax(fast)) if np.isfinite(fast).any() else PRE

        cap.set(cv2.CAP_PROP_POS_FRAMES,
                int(d["frame_120"][i]) - PRE + k)
        ok, fr = cap.read()
        if not ok:
            continue
        fr = draw(fr, kp[k], sc[k])

        b, pad = bx[k].astype(int), 150
        x1, y1 = max(0, b[0] - pad), max(0, b[1] - pad)
        x2, y2 = min(fr.shape[1], b[2] + pad), min(fr.shape[0], b[3] + pad)
        c = fr[y1:y2, x1:x2]
        if c.size == 0:
            continue
        H = 380
        c = cv2.resize(c, (int(H * c.shape[1] / c.shape[0]), H))
        c = np.vstack([banner(c.shape[1],
                              f"{cls[i]}  L={pL:.2f} R={pR:.2f}", h=28,
                              colour=(180, 255, 180), scale=0.5), c])
        tiles.append(c)

    cap.release()
    if not tiles:
        print(f"  {key}: no frames rendered"); continue

    w = min(t.shape[1] for t in tiles)
    sheet = np.hstack([t[:, :w] for t in tiles])
    head = banner(sheet.shape[1],
                  f"{key}   ->  called {info['handedness']}   "
                  f"margin {info['margin']:.3f}   n={info['n_samples']}", h=40,
                  colour=(0, 165, 255) if info["margin"] < 0.15 else (0, 255, 255),
                  scale=0.7)
    key_ = banner(sheet.shape[1],
                  "MAGENTA = left arm    CYAN = right arm    "
                  "bat arm = the extended one", h=28,
                  colour=(220, 220, 220), scale=0.5)
    sheet = np.vstack([head, key_, sheet])

    print("=" * 74)
    for sid, pL, pR in per_stroke:
        evidence.append({"player": key, "stroke": sid,
                         "L_speed": round(float(pL), 3),
                         "R_speed": round(float(pR), 3),
                         "faster": "L" if pL > pR else "R"})
    W = 1250
    disp = cv2.resize(sheet, (W, int(W * sheet.shape[0] / sheet.shape[1])))
    if cv2_imshow:
        cv2_imshow(disp)
    else:
        from IPython.display import Image, display
        _, buf = cv2.imencode(".jpg", disp); display(Image(data=buf.tobytes()))

print("\n" + "=" * 74)
print("NUMERIC EVIDENCE  (95th-percentile wrist speed, torso-normalised)")
print("=" * 74)
ev = pd.DataFrame(evidence)
if len(ev):
    print(ev.to_string(index=False))
    print("\nper-player agreement across sampled strokes:")
    agg = ev.groupby("player").faster.value_counts().unstack(fill_value=0)
    print(agg.to_string())
print("=" * 74)
print("""
WHAT TO LOOK FOR
 - The bat arm swings ACROSS the body and reaches furthest from the torso.
 - At peak speed the playing arm is extended; the free arm stays tucked.
 - Check the two CONTROLS (game_1__right, test_1__right) first. If they don't
    read clearly as right-handed, the visualisation is at fault, not the data.

THEN
  Set OVERRIDE in the previous cell for anything called wrong, e.g.

      OVERRIDE = {"test_5__right": "R"}

  and re-run it to rewrite players.json.
""")

Output hidden; open in https://colab.research.google.com to view.

In [15]:
# =============================================================================
# CELL 11 - INDEPENDENT HANDEDNESS CROSS-CHECK  (serve ball-toss signal)
#
# Cell 8 decides handedness from WRIST SPEED. This cell uses a completely
# different physical cue:
#
#     On a serve, the FREE hand tosses the ball upward while the PLAYING hand
#     stays low, holding the bat below the table line.
#
# So the wrist that rises highest above the shoulders before contact is the
# NON-playing hand. That gives an independent vote on handedness - and because
# it's driven by vertical position rather than speed, it fails in different
# ways than the velocity signal does.
#
# Where the two agree, the call is solid. Where they disagree, the player
# needs a human look.
#
# Outputs a text table - no images needed. Runs in seconds.
# =============================================================================

BASE = "/content/drive/MyDrive/tt_coach"

import json
from pathlib import Path
import numpy as np
import pandas as pd

BASE    = Path(BASE)
META    = BASE / "derived/meta"
POSEDIR = BASE / "derived/pose"

L_SHO, R_SHO = 5, 6
L_WRI, R_WRI = 9, 10
L_HIP, R_HIP = 11, 12

players = json.loads((META / "players.json").read_text())
folds   = json.loads((META / "folds.json").read_text())
PRE     = folds["window"]["pre"]          # contact sits at index PRE

VIDEO_IDS = sorted({v["video_id"] for v in players.values()},
                   key=lambda v: (v.split("_")[0], int(v.split("_")[1])))


def toss_heights(kp, sc, conf=0.4):
    """Max height of each wrist above the shoulder line, pre-contact.
    Image coords: y grows downward, so height = shoulder_y - wrist_y."""
    pre = slice(0, PRE)                       # everything before contact
    sho_y = (kp[pre, L_SHO, 1] + kp[pre, R_SHO, 1]) / 2

    torso = np.linalg.norm(
        (kp[:, L_SHO] + kp[:, R_SHO]) / 2 - (kp[:, L_HIP] + kp[:, R_HIP]) / 2,
        axis=-1)
    torso = np.median(torso[torso > 1]) if (torso > 1).any() else np.nan
    if not np.isfinite(torso) or torso <= 0:
        return np.nan, np.nan

    out = []
    for w in (L_WRI, R_WRI):
        h = (sho_y - kp[pre, w, 1]) / torso    # positive = above shoulders
        h = h[sc[pre, w] > conf]
        out.append(float(np.nanmax(h)) if len(h) else np.nan)
    return out[0], out[1]


rows = []
for vid in VIDEO_IDS:
    p = POSEDIR / f"{vid}.npz"
    if not p.exists():
        continue
    d = np.load(p, allow_pickle=True)
    KP, SC, DET = d["keypoints"], d["scores"], d["detected"]
    sides, cls = d["side"], d["shot_class"]

    for pi, side in enumerate(["left", "right"]):
        key = f"{vid}__{side}"
        sel = np.where((sides == side) & (cls == "serve") & DET[:, pi])[0]

        hl, hr = [], []
        for i in sel:
            a, b = toss_heights(KP[i, pi].astype(np.float32),
                                SC[i, pi].astype(np.float32))
            if np.isfinite(a) and np.isfinite(b):
                hl.append(a); hr.append(b)

        if len(hl) < 2:
            toss_hand, toss_margin = "?", np.nan
        else:
            mL, mR = float(np.median(hl)), float(np.median(hr))
            # the wrist that rises higher is the FREE hand -> playing hand is the other
            toss_hand = "R" if mL > mR else "L"
            denom = max(abs(mL), abs(mR))
            toss_margin = abs(mL - mR) / denom if denom > 0 else np.nan

        vel = players[key]["handedness"]
        rows.append({
            "player": key,
            "fold": players[key]["fold"],
            "n_serves": len(hl),
            "velocity": vel,
            "vel_margin": players[key]["margin"],
            "toss": toss_hand,
            "toss_margin": round(toss_margin, 3) if np.isfinite(toss_margin) else np.nan,
            "agree": "OK" if toss_hand == vel else ("--" if toss_hand == "?" else "CONFLICT"),
        })

df = pd.DataFrame(rows)

print("=" * 84)
print("HANDEDNESS: velocity signal  vs  serve-toss signal")
print("=" * 84)
print(df.to_string(index=False))

dec = df[df.toss != "?"]
n_ok = (dec.agree == "OK").sum()
print("\n" + "=" * 84)
print(f"  decidable: {len(dec)}/{len(df)}   agree: {n_ok}/{len(dec)} "
      f"({100*n_ok/max(len(dec),1):.0f}%)")

conf = df[df.agree == "CONFLICT"]
if len(conf):
    print(f"\n  CONFLICTS - two independent signals disagree:")
    print(conf.to_string(index=False))
    print("\n  For each, trust whichever signal has the larger margin, or")
    print("  look at the video. A conflict where BOTH margins are small")
    print("  usually means noisy pose for that player.")
else:
    print("\n  No conflicts - both signals agree everywhere they can decide.")

nod = df[df.toss == "?"]
if len(nod):
    print(f"\n  undecidable (too few clean serves): {list(nod.player)}")

print("\n" + "=" * 84)
print("SUGGESTED OVERRIDE")
print("=" * 84)
sug = {}
for _, r in df.iterrows():
    if r.agree == "CONFLICT":
        # prefer the signal with the stronger margin
        if np.isfinite(r.toss_margin) and r.toss_margin > r.vel_margin:
            sug[r.player] = r.toss
if sug:
    print("OVERRIDE = {")
    for k, v in sug.items():
        print(f'    "{k}": "{v}",')
    print("}")
    print("\n  ^ paste into cell 9 and re-run it")
else:
    print("OVERRIDE = {}   # nothing to change on this evidence")

lefties = df[df.velocity == "L"]
print(f"\n  current left-handers ({len(lefties)}): {list(lefties.player)}")
print("  (source paper reports 3 left-handed players, all in test videos)")
print("=" * 84)

HANDEDNESS: velocity signal  vs  serve-toss signal
       player fold  n_serves velocity  vel_margin toss  toss_margin    agree
 game_1__left    A        11        R       0.355    R        0.432       OK
game_1__right    A        13        R       0.530    L        0.232 CONFLICT
 game_2__left    B        38        R       0.452    R        0.248       OK
game_2__right    B        48        R       0.453    R        1.887       OK
 game_3__left    C        21        R       0.507    L        0.053 CONFLICT
game_3__right    C         9        R       0.400    L        0.442 CONFLICT
 game_4__left    D        11        R       0.300    L        0.940 CONFLICT
game_4__right    D        17        R       0.415    L        0.427 CONFLICT
 game_5__left    E        28        R       0.199    L        1.433 CONFLICT
game_5__right    E        24        R       0.411    R        1.759       OK
 test_1__left    F         2        R       0.312    L        0.110 CONFLICT
test_1__right    F       

In [18]:
# =============================================================================
# REBUILD players.json  -  self-contained, ignores all session state
#
# Recomputes handedness from the .npz pose cache and writes players.json from
# scratch. Safe to run at any time, in any order, after a restart - it reads
# only from disk, so no earlier cell can contaminate it.
#
# This replaces cells 8 + 9. Use it instead of them from now on.
# =============================================================================

BASE = "/content/drive/MyDrive/tt_coach"

# Manual corrections, applied fresh each run. Empty = pure derivation.
OVERRIDE = {}

import json
from pathlib import Path
import numpy as np
import pandas as pd

BASE    = Path(BASE)
META    = BASE / "derived/meta"
POSEDIR = BASE / "derived/pose"

L_SHO, R_SHO = 5, 6
L_WRI, R_WRI = 9, 10
L_HIP, R_HIP = 11, 12

folds = json.loads((META / "folds.json").read_text())

def load(stem):
    p = META / f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META / f"{stem}.csv")

strokes = load("strokes")
VIDEO_IDS = sorted(strokes.video_id.unique(),
                   key=lambda v: (v.split("_")[0], int(v.split("_")[1])))


def wrist_stats(kp, sc, conf=0.5):
    """Torso-normalised 95th-percentile speed of each wrist."""
    torso = np.linalg.norm(
        (kp[:, L_SHO] + kp[:, R_SHO]) / 2 - (kp[:, L_HIP] + kp[:, R_HIP]) / 2,
        axis=-1)
    torso = np.median(torso[torso > 1]) if (torso > 1).any() else 1.0

    out = []
    for w in (L_WRI, R_WRI):
        xy, s = kp[:, w].astype(np.float32), sc[:, w]
        good = s > conf
        if good.sum() < 10:
            out.append(np.nan); continue
        v = np.linalg.norm(np.diff(xy, axis=0), axis=-1)
        vg = v[good[:-1] & good[1:]]
        out.append(float(np.percentile(vg, 95) / torso) if len(vg) else np.nan)
    return out


players, rows = {}, []

for vid in VIDEO_IDS:
    p = POSEDIR / f"{vid}.npz"
    if not p.exists():
        print(f"  {vid}: no pose cache, skipped"); continue
    d = np.load(p, allow_pickle=True)
    KP, SC, DET, sides = d["keypoints"], d["scores"], d["detected"], d["side"]

    for pi, side in enumerate(["left", "right"]):
        idx = np.where((sides == side) & DET[:, pi])[0]
        L, R = [], []
        for i in idx:
            a, b = wrist_stats(KP[i, pi].astype(np.float32),
                               SC[i, pi].astype(np.float32))
            if np.isfinite(a) and np.isfinite(b):
                L.append(a); R.append(b)

        if len(L) < 3:
            hand, margin, method = "R", 0.0, "default_no_data"
        else:
            mL, mR = float(np.median(L)), float(np.median(R))
            if max(mL, mR) <= 0:
                hand, margin, method = "R", 0.0, "default_no_signal"
            else:
                hand = "L" if mL > mR else "R"
                margin = abs(mL - mR) / max(mL, mR)
                method = "wrist_velocity_p95"

        key = f"{vid}__{side}"
        if key in OVERRIDE:
            hand, method = OVERRIDE[key].upper(), "manual_override"

        sv = strokes[(strokes.video_id == vid) & (strokes.side == side)]
        players[key] = {
            "video_id": vid, "side": side,
            "handedness": hand,
            "margin": round(margin, 3),
            "n_samples": len(L),
            "fold": folds["video2fold"][vid],
            "n_strokes": int(len(sv)),
            "class_counts": sv.shot_class.value_counts().to_dict(),
            "method": method,
            "note": "",
        }
        rows.append({"player": key, "fold": folds["video2fold"][vid],
                     "hand": hand, "margin": round(margin, 3),
                     "n": len(L), "method": method})

(META / "players.json").write_text(json.dumps(players, indent=2))

sm = pd.DataFrame(rows)
print(sm.to_string(index=False))

lefties = sm[sm.hand == "L"]
n_train = sum(1 for k in lefties.player if k.startswith("game"))

print("\n" + "=" * 70)
print(f"  left-handed: {len(lefties)} / {len(sm)}")
for _, r in lefties.iterrows():
    print(f"    {r.player:<20} fold {r.fold}  margin {r.margin:.3f}  [{r.method}]")
print(f"\n  in TRAINING videos : {n_train}")
print(f"  in TEST videos     : {len(lefties) - n_train}")
print("\n  The source paper reports 3 left-handed players, all in test videos.")
print("  Zero left-handers in game_* is a strong sanity signal: the derivation")
print("  has no knowledge of the train/test split, so that separation is not")
print("  something a broken method would produce by chance.")

overridden = sm[sm.method == "manual_override"]
print(f"\n  manual overrides applied: {len(overridden)}"
      + (f" -> {list(overridden.player)}" if len(overridden) else ""))
print("=" * 70)
print(f"\n-> {META / 'players.json'}")

       player fold hand  margin   n             method
 game_1__left    A    R   0.349  75 wrist_velocity_p95
game_1__right    A    R   0.587  86 wrist_velocity_p95
 game_2__left    B    R   0.389 199 wrist_velocity_p95
game_2__right    B    R   0.464 188 wrist_velocity_p95
 game_3__left    C    R   0.485  81 wrist_velocity_p95
game_3__right    C    R   0.352  71 wrist_velocity_p95
 game_4__left    D    R   0.231  80 wrist_velocity_p95
game_4__right    D    R   0.438  89 wrist_velocity_p95
 game_5__left    E    R   0.217 124 wrist_velocity_p95
game_5__right    E    R   0.448 124 wrist_velocity_p95
 test_1__left    F    R   0.298  44 wrist_velocity_p95
test_1__right    F    R   0.434  40 wrist_velocity_p95
 test_2__left    G    R   0.347  10 wrist_velocity_p95
test_2__right    G    R   0.672  19 wrist_velocity_p95
 test_3__left    G    R   0.441  14 wrist_velocity_p95
test_3__right    G    L   0.291  10 wrist_velocity_p95
 test_4__left    F    R   0.136  33 wrist_velocity_p95
test_4__ri